# Website Bronze Ingestion from S3

Ingested raw website data (sessions, page views, events) from the Airbyte S3 landing zone into bronze Delta tables. Validated the ingested data and checked for duplicates before moving to the silver layer.

## Exploring the raw S3 landing zone

First, checked what Airbyte actually landed in S3 for the three website streams. Listed the files in each folder to confirm data was present before reading anything.

In [0]:
base_path = "s3://voyage-data-plateform-yuvaraj-india/raw/airbyte/voyage_raw/"

website_folders = [
    "web_sessions",
    "web_page_views",
    "web_events",
]

for folder in website_folders:
    path = f"{base_path}{folder}/"
    print(f"\n===== {folder} =====")
    files = dbutils.fs.ls(path)
    display(files)

## Inspecting raw JSON structure

Now that the folders existed and had data, read a sample from each to understand the JSON schema. Needed to see what was nested inside `_airbyte_data` and what Airbyte metadata columns were available — this determined how to flatten everything in the next step.

In [0]:
for folder in website_folders:
    path = f"{base_path}{folder}/"

    print(f"\n===== {folder} =====")

    df = spark.read.json(path)

    print("Row count:", df.count())

    df.printSchema()

    display(df.limit(3))

## Flattening and writing to bronze Delta tables

The raw JSON nested the actual payload inside `_airbyte_data`. Flattened that out, kept the Airbyte metadata columns (`_airbyte_extracted_at`, `_airbyte_generation_id`, `_airbyte_meta`, `_airbyte_raw_id`), and wrote each stream to `voyage.bronze` as a Delta table. Overwrote schema since this was the initial load.

In [0]:
from pyspark.sql import functions as F


BASE_PATH = (
    "s3://voyage-data-plateform-yuvaraj-india/"
    "raw/airbyte/voyage_raw/"
)

WEBSITE_STREAMS = {
    "web_sessions": "website_sessions",
    "web_page_views": "website_page_views",
    "web_events": "website_events",
}


def read_airbyte_stream(stream_name: str):
    path = f"{BASE_PATH}{stream_name}/"

    raw_df = spark.read.json(path)

    flattened_df = raw_df.select(
        "_airbyte_data.*",
        "_airbyte_extracted_at",
        "_airbyte_generation_id",
        "_airbyte_meta",
        "_airbyte_raw_id",
    )

    return flattened_df
for stream_name, table_name in WEBSITE_STREAMS.items():

    print(f"\n===== {stream_name} =====")

    df = read_airbyte_stream(stream_name)

    print(f"Raw Bronze rows: {df.count()}")
    print(f"Writing: voyage.bronze.{table_name}")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"voyage.bronze.{table_name}")
    )

    print(f"Completed: voyage.bronze.{table_name}")



## Spot-checking the ingested sessions

Bronze tables were written — did a quick `SELECT * LIMIT 10` on `website_sessions` to make sure the flattening worked and the columns looked right.

In [0]:
%sql
select *from voyage.bronze.website_sessions limit 10;

## Spot-checking the ingested page views

Same check on `website_page_views` — confirmed the page-level fields (page_path, referrer, duration) came through correctly from the flatten.

In [0]:
%sql
select *from voyage.bronze.website_page_views limit 10;

## Spot-checking the ingested events

And the same on `website_events` — verified event_name and event_value were populated as expected after the flatten.

In [0]:
%sql
select*from voyage.bronze.website_events limit 10;


## Checking for duplicate events

Airbyte extractions can land the same record more than once, especially on re-syncs. Compared total row count vs distinct `event_id` — if there was a gap, the silver layer needed to deduplicate.

In [0]:
%sql
SELECT count(*), count(distinct event_id) FROM voyage.bronze.website_events;

## Checking for duplicate sessions

Same check on `website_sessions` — compared total rows vs distinct `session_id` to see if the silver layer needed to deduplicate.

In [0]:
%sql
select count(*), count(distinct session_id) from voyage.bronze.website_sessions
;

## Checking for duplicate page views

Last one — same row count vs distinct `page_view_id` check on `website_page_views`. These three dedup checks told the silver layer (notebook 06) which business keys to deduplicate on.

In [0]:
%sql
select count(*), count(distinct page_view_id) from voyage.bronze.website_page_views;